Dataset 1:  A random sample of 500 ad explanations per month for each platform (Facebook, Instagram, Twitter, and YouTube), totaling 2,000 ad explanations per month.

This is the main dataset for both Study 1 and Study 2. 

Platforms: Facebook, Instagram, Twitter, and YouTube.
Timeframe: From 2024-08-29 to the present, From 2017 to present for Facebook
Geographic Scope: Germany-based users only.

Total: Totalling 192K explanations for Facebook and 42K explanations for the other platforms. 

In [1]:

%env AWS_ACCESS_KEY_ID=xxxx
%env AWS_SECRET_ACCESS_KEY=xxxx
%env AWS_SESSION_TOKEN=xxxx

env: AWS_ACCESS_KEY_ID=xxxx
env: AWS_SECRET_ACCESS_KEY=xxxx
env: AWS_SESSION_TOKEN=xxxx


In [88]:
import awswrangler as wr
import pyarrow as pa
import pandas as pd

In [ ]:
schema = pa.schema([
    ("provider_data.rawlog_id", pa.string()),
    ("provider_data.user.id", pa.string()),
    ("provider_data.user.country", pa.string()),
    ("provider_data.waist", pa.string()),
    ("advertisement.details.advert_id", pa.string()),
    ("advertiser.details.advertiser_id", pa.string()),
    ("advertiser.details.advertiser_real_name", pa.string()),
    ("advertiser.details.advertiser_internal_name", pa.string()),
    ("advertiser.details.advertiser_website", pa.string()),
    ("targeting_categories", pa.string()),
    ("platform", pa.string()),
    ("day", pa.string()),
    ("month", pa.string()),
    ("year", pa.string()),
    ("country", pa.string()),
])

country = 'DE'
year = '2025'
month = '2'

def partition_filter(x):
    return (
        x['country'] == country and 
        (
            x['year'] == year and 
            x['month'] == month 
        ) 
    )

In [90]:
df = wr.s3.read_parquet(
  path='s3://platform-impressions-export/parquet/impressions/',
  dataset=True,
  partition_filter=partition_filter,
  columns=[
    'provider_data.user.id', 
    'provider_data.waist', 
    'provider_data.rawlog_id',
    'advertisement.details.advert_id',
    'advertiser.details.advertiser_id',
    'advertiser.details.advertiser_real_name',
    'advertiser.details.advertiser_internal_name',
    'advertiser.details.advertiser_website',
    'targeting_categories',
    'platform', 
    'day', 
    'month', 
    'year', 
    'country'
  ],
  schema=schema
)

In [91]:
df_facebook = df[df['platform'] == 'Facebook']
df_instagram = df[df['platform'] == 'Instagram']
df_twitter = df[df['platform'] == 'Twitter']
df_youtube = df[df['platform'] == 'Youtube']

In [92]:
from pathlib import Path

for platform in ['Facebook', 'Instagram', 'Twitter', 'Youtube'] :
  output_dir = Path(f"study-1/platform={platform}/country={country}/year={year}/month={month}")
  output_dir.mkdir(parents=True, exist_ok=True)


In [93]:
# write for each platform

df_facebook.sample(500).to_csv(f"study-1/platform=Facebook/country={country}/year={year}/month={month}/impressions.csv")
df_instagram.sample(500).to_csv(f"study-1/platform=Instagram/country={country}/year={year}/month={month}/impressions.csv")
df_twitter.sample(500).to_csv(f"study-1/platform=Twitter/country={country}/year={year}/month={month}/impressions.csv")
df_youtube.sample(500).to_csv(f"study-1/platform=Youtube/country={country}/year={year}/month={month}/impressions.csv")